In [1]:
import torch, torchvision
import torch.nn as nn, torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader,Subset

t = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Grayscale(3),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

#bit of the nasty to make it faster ☠️
trd = DataLoader(Subset(torchvision.datasets.MNIST('./data',True,transform=t,download=True), range(200)), 64, True)
ted = DataLoader(Subset(torchvision.datasets.MNIST('./data',False,transform=t,download=True), range(50)), 1000)


Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%

Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw



In [2]:
class AlexNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.feats = nn.Sequential(
            nn.Conv2d(3,64,11,4,2), nn.ReLU(), nn.MaxPool2d(3,2),
            nn.Conv2d(64,192,5,padding=2), nn.ReLU(), nn.MaxPool2d(3,2),
            nn.Conv2d(192,384,3,padding=1), nn.ReLU(),
            nn.Conv2d(384,256,3,padding=1), nn.ReLU(),
            nn.Conv2d(256,256,3,padding=1), nn.ReLU(), nn.MaxPool2d(3,2)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(), nn.Linear(256*6*6,4096),
            nn.Dropout(), nn.Linear(4096,4096), nn.ReLU(),
            nn.Linear(4096,10)
        )
    def forward(self, x):
        x = self.feats(x)
        x = x.view(x.size(0),-1)
        x = self.classifier(x)
        return x

model = AlexNet()
crit  = nn.CrossEntropyLoss()
opt   = optim.Adam(model.parameters(), lr=0.001)

In [7]:
for e in range(1):
    model.train()
    rl = 0
    for i, (x, y) in enumerate(trd):
        opt.zero_grad()
        l = crit(model(x), y)
        l.backward()
        opt.step()
        rl += l.item()
        if i % 100 == 0:
            print(f'E{e+1} B{i+1} | Loss: {rl:.3f}')
            rl = 0

E1 B1 | Loss: 2.322


In [8]:
model.eval()
c = n = 0
with torch.no_grad():
    for x, y in ted:
        c += (model(x).argmax(1)==y).sum().item()
        n += len(y)
print(f'Accuracy: {100*c/n:.2f}%')

Accuracy: 18.00%
